In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import json


In [4]:
# Try reading with absolute path
file_path = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/cleaned_data/preprocess_filtered.csv"
print(f"File exists: {os.path.exists(file_path)}")


File exists: True


In [5]:
df = pd.read_csv(file_path)

In [6]:
df.columns

Index(['stepPreState', 'stepPostState', 'stepHintGiven', 'currentProblem',
       'currentProblemType', 'currentProblemDescription',
       'currentProblemMetaData', 'sAssertion', 'cleanedStates', 'stateIDs'],
      dtype='object')

In [7]:
# create pivot table from currentProblem jsonl entry
df["currentProblem"].value_counts().sort_index()



currentProblem
2.2     2
2.3     7
2.4    18
2.5     8
2.6    10
2.7     8
3.2     2
3.3    18
3.4    34
3.5    23
3.6    60
3.7     9
4.1     4
4.2    10
4.3    15
4.4    14
4.5    54
4.6    37
4.7    18
5.1     3
5.2     1
5.3     7
5.4    16
5.5    14
5.6    45
5.7     9
6.1     3
6.3    10
6.4    42
6.5    11
6.6    38
6.7     8
Name: count, dtype: int64

In [45]:
# Convert to string first, then extract level
df['level'] = df['currentProblem'].astype(str).str.split('.').str[0].astype(int)
level_counts = df.groupby('level').size().sort_index()
print(level_counts)

level
2     53
3    146
4    152
5     95
6    112
dtype: int64


In [9]:
df.shape

(558, 11)

In [10]:
df.head()

,stepPreState,stepPostState,stepHintGiven,currentProblem,currentProblemType,currentProblemDescription,currentProblemMetaData,sAssertion,cleanedStates,stateIDs,instances_per_level
0,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive (F+G) working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",(F+G),"""((F+G)>H)"",""(I+F)"",""(-I*J)"",""-I"",""F""","[3, 19, 33, 42, 43]",2
1,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive -I working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",-I,"""((F+G)>H)"",""(I+F)"",""(-I*J)""","[3, 19, 33]",2
2,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive N working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",N,"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-K+L)"",""(...","[17, 59, 77, 79, 98, 100]",7
3,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K""","[17, 77, 98, 100]",7
4,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-(-K+L)+(...","[8, 17, 21, 35, 77, 98, 100]",7


In [11]:
df["stepPreState"][1]

'((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3;0;Given]/H'

In [12]:
def clean_and_quote_states(prestate_str):
    # remove trailing /goal
    prestate_str = prestate_str.split("/")[0]
    # split into states
    states = prestate_str.split("],")
    states = [s.strip() + "]" if not s.strip().endswith("]") else s.strip() for s in states]
    
    cleaned = []
    for s in states:
        # remove everything inside brackets [ ... ]
        formula = re.sub(r"\[.*?\]", "", s).strip()
        # wrap in quotes
        cleaned.append(f'"{formula}"')
    
    # join with commas
    return ",".join(cleaned)

In [13]:
# Apply function to each row
df["cleanedStates"] = df["stepPreState"].apply(clean_and_quote_states)
df["cleanedStates"].head()

0                "((F+G)>H)","(I+F)","(-I*J)","-I","F"
1                         "((F+G)>H)","(I+F)","(-I*J)"
2    "((-K+L)>(M*N))","(K>O)","-O","-K","(-K+L)","(...
3                   "((-K+L)>(M*N))","(K>O)","-O","-K"
4    "((-K+L)>(M*N))","(K>O)","-O","-K","(-(-K+L)+(...
Name: cleanedStates, dtype: object

In [14]:
# convert states to ids
mapping_dir = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/map" 
def map_states_to_ids(row):
    problem_num = row["currentProblem"]
    states = [s.strip().strip('"') for s in row["cleanedStates"].split(",")]  # remove quotes for lookup

    mapping_file = os.path.join(mapping_dir, f"mapPropositions_{problem_num}.json")
    if not os.path.exists(mapping_file):
        print(f"⚠️ Mapping file not found for problem {problem_num}")
        return []

    with open(mapping_file) as f:
        mapping = json.load(f)

    ids = []
    for s in states:
        if s in mapping:
            ids.append(mapping[s]["id"])
        else:
            ids.append(None)  # or skip if you prefer
    return ids



In [15]:
df["stateIDs"] = df.apply(map_states_to_ids, axis=1)
df["stateIDs"].head()

0             [3, 33, 19, 42, 43]
1                     [3, 33, 19]
2       [17, 77, 100, 98, 59, 79]
3               [17, 77, 100, 98]
4    [17, 77, 100, 98, 35, 8, 21]
Name: stateIDs, dtype: object

In [16]:
# Drop rows where any None appears in stateIDs
df = df[~df["stateIDs"].apply(lambda x: any(i is None for i in x))].reset_index(drop=True)

# Check result
print(df[["cleanedStates", "stateIDs"]])


                                         cleanedStates  \
0                "((F+G)>H)","(I+F)","(-I*J)","-I","F"   
1                         "((F+G)>H)","(I+F)","(-I*J)"   
2    "((-K+L)>(M*N))","(K>O)","-O","-K","(-K+L)","(...   
3                   "((-K+L)>(M*N))","(K>O)","-O","-K"   
4    "((-K+L)>(M*N))","(K>O)","-O","-K","(-(-K+L)+(...   
..                                                 ...   
553  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
554  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
555  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
556  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
557  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   

                            stateIDs  
0                [3, 33, 19, 42, 43]  
1                        [3, 33, 19]  
2          [17, 77, 100, 98, 59, 79]  
3                  [17, 77, 100, 98]  
4       [17, 77, 100, 98, 35, 8, 21]  
..                               ...  
553          [33, 25, 31, 3, 3

In [17]:
# Sort stateIDs in ascending order for each row
df["stateIDs"] = df["stateIDs"].apply(lambda x: sorted(x))

# Check result
print(df[["cleanedStates", "stateIDs"]])

                                         cleanedStates  \
0                "((F+G)>H)","(I+F)","(-I*J)","-I","F"   
1                         "((F+G)>H)","(I+F)","(-I*J)"   
2    "((-K+L)>(M*N))","(K>O)","-O","-K","(-K+L)","(...   
3                   "((-K+L)>(M*N))","(K>O)","-O","-K"   
4    "((-K+L)>(M*N))","(K>O)","-O","-K","(-(-K+L)+(...   
..                                                 ...   
553  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
554  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
555  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
556  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
557  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   

                            stateIDs  
0                [3, 19, 33, 42, 43]  
1                        [3, 19, 33]  
2          [17, 59, 77, 79, 98, 100]  
3                  [17, 77, 98, 100]  
4       [8, 17, 21, 35, 77, 98, 100]  
..                               ...  
553          [3, 25, 31, 33, 3

In [18]:
# Alternatively, ensure uniqueness by converting to tuples
df = df[~df["stateIDs"].duplicated(keep="first")].reset_index(drop=True)

# Check result
print(df[["cleanedStates", "stateIDs"]])

                                         cleanedStates  \
0                "((F+G)>H)","(I+F)","(-I*J)","-I","F"   
1                         "((F+G)>H)","(I+F)","(-I*J)"   
2    "((-K+L)>(M*N))","(K>O)","-O","-K","(-K+L)","(...   
3                   "((-K+L)>(M*N))","(K>O)","-O","-K"   
4    "((-K+L)>(M*N))","(K>O)","-O","-K","(-(-K+L)+(...   
..                                                 ...   
553  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
554  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
555  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
556  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   
557  "(B=-J)","(-N+J)","(B+-N)","((B>-J)*(-J>B))","...   

                            stateIDs  
0                [3, 19, 33, 42, 43]  
1                        [3, 19, 33]  
2          [17, 59, 77, 79, 98, 100]  
3                  [17, 77, 98, 100]  
4       [8, 17, 21, 35, 77, 98, 100]  
..                               ...  
553          [3, 25, 31, 33, 3

In [19]:
df.head()

,stepPreState,stepPostState,stepHintGiven,currentProblem,currentProblemType,currentProblemDescription,currentProblemMetaData,sAssertion,cleanedStates,stateIDs,instances_per_level
0,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive (F+G) working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",(F+G),"""((F+G)>H)"",""(I+F)"",""(-I*J)"",""-I"",""F""","[3, 19, 33, 42, 43]",2
1,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive -I working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",-I,"""((F+G)>H)"",""(I+F)"",""(-I*J)""","[3, 19, 33]",2
2,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive N working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",N,"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-K+L)"",""(...","[17, 59, 77, 79, 98, 100]",7
3,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K""","[17, 77, 98, 100]",7
4,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-(-K+L)+(...","[8, 17, 21, 35, 77, 98, 100]",7


In [20]:
df['sAssertion'] = df['stepHintGiven'].str.replace('^Try to derive ', '', regex=True).str.replace(' working forward.$', '', regex=True)
print("filtered file shape after adding the stepHintGiven_cleaned column: ", df.shape)
        # remove all the rows where stepHintGiven = Try to derive the conclusion.
df = df[df['stepHintGiven'] != "Try to derive the conclusion."]
print("filtered file shape after removing all the rows where stepHintGiven = 'Try to derive the conclusion.': ", df.shape)

filtered file shape after adding the stepHintGiven_cleaned column:  (558, 11)
filtered file shape after removing all the rows where stepHintGiven = 'Try to derive the conclusion.':  (558, 11)


In [21]:
df.head()

,stepPreState,stepPostState,stepHintGiven,currentProblem,currentProblemType,currentProblemDescription,currentProblemMetaData,sAssertion,cleanedStates,stateIDs,instances_per_level
0,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive (F+G) working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",(F+G),"""((F+G)>H)"",""(I+F)"",""(-I*J)"",""-I"",""F""","[3, 19, 33, 42, 43]",2
1,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive -I working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",-I,"""((F+G)>H)"",""(I+F)"",""(-I*J)""","[3, 19, 33]",2
2,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive N working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",N,"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-K+L)"",""(...","[17, 59, 77, 79, 98, 100]",7
3,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K""","[17, 77, 98, 100]",7
4,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-(-K+L)+(...","[8, 17, 21, 35, 77, 98, 100]",7


In [22]:
# check if there are any occurence of '#' in any of the columns
df.apply(lambda x: x.astype(str).str.contains('#').any())
# how many rows have '#' in any of the columns
df.apply(lambda x: x.astype(str).str.contains('#').sum())
# print the stepPreState where '#' is present in any of the columns
df[df.apply(lambda x: x.astype(str).str.contains('#').any(), axis=1)]['stepPreState']



Series([], Name: stepPreState, dtype: object)

In [23]:
print("\nPivot Table (currentProblem as rows, currentProblemType as columns):")
pivot = df.pivot_table(index='currentProblem', columns='currentProblemType', aggfunc='size', fill_value=0)
pd.set_option('display.max_rows', None)
print(pivot)


Pivot Table (currentProblem as rows, currentProblemType as columns):
currentProblemType  PS
currentProblem        
2.2                  2
2.3                  7
2.4                 18
2.5                  8
2.6                 10
2.7                  8
3.2                  2
3.3                 18
3.4                 34
3.5                 23
3.6                 60
3.7                  9
4.1                  4
4.2                 10
4.3                 15
4.4                 14
4.5                 54
4.6                 37
4.7                 18
5.1                  3
5.2                  1
5.3                  7
5.4                 16
5.5                 14
5.6                 45
5.7                  9
6.1                  3
6.3                 10
6.4                 42
6.5                 11
6.6                 38
6.7                  8


In [24]:
df.shape

(558, 11)

In [25]:
df.columns

Index(['stepPreState', 'stepPostState', 'stepHintGiven', 'currentProblem',
       'currentProblemType', 'currentProblemDescription',
       'currentProblemMetaData', 'sAssertion', 'cleanedStates', 'stateIDs',
       'instances_per_level'],
      dtype='object')

In [26]:
df['stepHintGiven'].value_counts()
# remove where backward hint given is 
# Remove rows where stepHintGiven contains 'backward'
df = df[~df['stepHintGiven'].str.contains('backward', case=False, na=False)]

In [27]:
df.shape

(558, 11)

In [28]:
df['currentProblem'].value_counts().sort_values(ascending=False)

currentProblem
3.6    60
4.5    54
5.6    45
6.4    42
6.6    38
4.6    37
3.4    34
3.5    23
4.7    18
3.3    18
2.4    18
5.4    16
4.3    15
4.4    14
5.5    14
6.5    11
2.6    10
4.2    10
6.3    10
5.7     9
3.7     9
6.7     8
2.7     8
2.5     8
5.3     7
2.3     7
4.1     4
5.1     3
6.1     3
3.2     2
2.2     2
5.2     1
Name: count, dtype: int64

In [29]:
df['stepPreState'].nunique()

558

In [30]:
df['stepHintGiven'].nunique()

261

In [31]:
df.groupby('currentProblem')['stepHintGiven'].nunique()

currentProblem
2.2     2
2.3     5
2.4     9
2.5     5
2.6     5
2.7     7
3.2     2
3.3     9
3.4    11
3.5    11
3.6    21
3.7     7
4.1     4
4.2     7
4.3    10
4.4     8
4.5    15
4.6    18
4.7    13
5.1     3
5.2     1
5.3     6
5.4     8
5.5     8
5.6    20
5.7     7
6.1     3
6.3     7
6.4    24
6.5     6
6.6    18
6.7     6
Name: stepHintGiven, dtype: int64

In [32]:
df['stepHintGiven'].value_counts()

stepHintGiven
Try to derive Y working forward.                    11
Try to derive (B+F) working forward.                 8
Try to derive (-D>F) working forward.                8
Try to derive (-S+Q) working forward.                8
Try to derive S working forward.                     7
Try to derive --(H+F) working forward.               7
Try to derive -D working forward.                    6
Try to derive (-C+-D) working forward.               6
(-C>-P)                                              6
Try to derive G working forward.                     6
Try to derive K working forward.                     5
(P>C)                                                5
-(-S+Y)                                              5
((-Y+T)>(X+S))                                       5
F                                                    5
(-M>-K)                                              5
G                                                    5
-F                                                 

In [33]:
df.groupby('stepPreState')['stepHintGiven'].count()

stepPreState
((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[3;0;Given],-K[4;2,3;Modus Tollens],(-(-K+L)+(M*N))[5;1;Implication],((--K*-L)+(M*N))[6;5;DeMorgan's Law],((K*-L)+(M*N))[7;6;Double Negation],(-K+L)[8;4;Addition]/N                                                                                                                                                                                                                                 1
((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[3;0;Given],-K[4;2,3;Modus Tollens],(-(-K+L)+(M*N))[5;1;Implication],((--K*-L)+(M*N))[6;5;DeMorgan's Law],((K*-L)+(M*N))[7;6;Double Negation]/N                                                                                                                                                                                                                                                      1
((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[3;0;Given],-K[4;2,3;Modus Tollens],(-(-K+L)+(M*N))[5;1;Implication]/N

In [34]:
df['sAssertion'].unique()

array(['(F+G)', '-I', 'N', '(-K+L)', '(M*N)', 'J', 'F', '(G*-H)', '-H',
       '(J+K)', '(K*O)', 'K', '(Q>P)', '-N', '(-O+L)', '(M*-N)', '(-Q+P)',
       'S', '(-T*S)', '(-R+N)', '-Q', '(-F*-K)', 'C', '(C+D)', '-B',
       '(B>U)', '(-B*H)', 'H', '(L+C)', '(S>D)', '(-S+Q)', '-S', 'Y',
       '((S*-Q)+Y)', '((--S*-Q)+Y)', '-D', '-F', '(-C+-D)', '-(-K+-Z)',
       '(-(-K+-Z)>-D)', '(C+F)', '(S>Y)', '(D+Y)', '-(S>Y)', 'D', '(I*Q)',
       '-(-S+Y)', '(--S*-Y)', '--S', '(S*-Y)', '(-S+T)', '-K', '(-R+P)',
       '(K>P)', '-O', '-P', '(R>P)', '(U>-W)', '--(U>K)', '(-W>A)', 'G',
       '-J', '-L', '(G*-J)', '(-L>-J)', '(-N+P)', '-G', '(K*L)', '(N>J)',
       '(-(K*L)+(N>J))', '((B+S)>(D+G))', '(D+G)', '(-D>--S)', '(-S>D)',
       '(-B>--S)', '(-D>S)', '(B+S)', '(B+F)', '((A+-D)>(B+F))', '(-D>F)',
       '((B+F)*G)', '(T>S)', '(-X>Y)', '((-Y+T)>(X+S))', '(-X>--Y)',
       '(-Y>X)', '--Z', '-W', '(X+S)', '(Y>S)', 'Z', '(Y+X)', '(-X>S)',
       '(Y>T)', '(H+-G)', 'P', '(--B>-H)', '(-G>A)', '(-G+

In [35]:
#add another column to the dataframe called "sPreState_length"
# count the , seprated values in the sPreState column
df['stepPreState_length'] = df['stepPreState'].str.count(',') + 1
df.head()

,stepPreState,stepPostState,stepHintGiven,currentProblem,currentProblemType,currentProblemDescription,currentProblemMetaData,sAssertion,cleanedStates,stateIDs,instances_per_level,stepPreState_length
0,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive (F+G) working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",(F+G),"""((F+G)>H)"",""(I+F)"",""(-I*J)"",""-I"",""F""","[3, 19, 33, 42, 43]",2,6
1,"((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...","((F+G)>H)[1;0;Given],(I+F)[2;0;Given],(-I*J)[3...",Try to derive -I working forward.,2.2,PS,"(F+G)>H,I+F,-I*J/H","SIMP,DS,ADD,MP",-I,"""((F+G)>H)"",""(I+F)"",""(-I*J)""","[3, 19, 33]",2,3
2,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive N working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",N,"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-K+L)"",""(...","[17, 59, 77, 79, 98, 100]",7,8
3,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K""","[17, 77, 98, 100]",7,5
4,"((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...","((-K+L)>(M*N))[1;0;Given],(K>O)[2;0;Given],-O[...",Try to derive (-K+L) working forward.,2.3,PS,"(-K+L)>(M*N),K>O,-O/N","MT,ADD,MP,SIMP",(-K+L),"""((-K+L)>(M*N))"",""(K>O)"",""-O"",""-K"",""(-(-K+L)+(...","[8, 17, 21, 35, 77, 98, 100]",7,8


In [36]:
df['stepPreState_length'].unique()

array([ 6,  3,  8,  5,  9,  4,  7, 10, 13, 11, 14, 12, 15, 16, 17, 19, 23,
       18,  2])